In [4]:
# ==============================================================================
# PCSK9 CRISPR gRNA Discovery Pipeline
# - Fetches GenBank locus from NCBI
# - Parses exon annotations
# - Scans both strands for SpCas9 PAM sites (NGG)
# - Applies transparent sequence filters
# - Ranks candidates with an honest heuristic score
#
# IMPORTANT:
# This script does NOT pretend to calculate validated Rule Set 2/3 or CFD scores.
# For validated scoring, export candidates and run them through crisprScore / crisprDesign.
# ==============================================================================

# Colab / notebook install
# !pip install biopython pandas -q

import re
import math
import pandas as pd
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

# -----------------------------
# NCBI configuration
# -----------------------------
Entrez.email = "your_real_email@example.com"  # Replace with your real email
Entrez.tool = "PCSK9_gRNA_pipeline"

ACCESSION = "NG_009061"   # Human PCSK9 genomic RefSeq locus
EXON_NUMBER = 1
MIN_GC = 40.0
MAX_GC = 65.0

# -----------------------------
# Helpers
# -----------------------------
def revcomp(seq: str) -> str:
    return str(Seq(seq).reverse_complement()).upper()

def gc_percent(seq: str) -> float:
    seq = seq.upper()
    return 100.0 * (seq.count("G") + seq.count("C")) / len(seq)

def has_poly_t(seq: str) -> bool:
    return "TTTT" in seq.upper()

def longest_homopolymer_run(seq: str) -> int:
    seq = seq.upper()
    best = 1
    current = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            current += 1
            best = max(best, current)
        else:
            current = 1
    return best

def heuristic_rank_score(spacer: str) -> float:
    """
    Transparent heuristic, not a published model.
    Higher is better.
    """
    gc = gc_percent(spacer)
    score = 100.0

    # Prefer moderate GC near 50%
    score -= abs(gc - 50.0) * 1.4

    # Penalize transcriptional terminator motifs and low complexity
    if has_poly_t(spacer):
        score -= 30.0

    run = longest_homopolymer_run(spacer)
    if run >= 5:
        score -= (run - 4) * 5.0

    # Mild preference for a G close to PAM-proximal end
    if spacer[-1] == "G":
        score += 2.0
    if spacer[-2] == "G":
        score += 1.0

    return round(max(0.0, min(100.0, score)), 2)

def fetch_genbank_record(accession: str):
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
    record = SeqIO.read(handle, "genbank")
    handle.close()
    return record

def get_exon_features(record):
    exons = [f for f in record.features if f.type == "exon"]
    if not exons:
        raise ValueError("No exon features found in the GenBank record.")
    exons = sorted(exons, key=lambda f: int(f.location.start))
    return exons

def extract_exon_sequence(record, exon_number: int):
    exons = get_exon_features(record)
    if exon_number < 1 or exon_number > len(exons):
        raise ValueError(f"Requested exon {exon_number} but only {len(exons)} exon feature(s) were found.")

    feature = exons[exon_number - 1]
    exon_seq = str(feature.extract(record.seq)).upper()
    exon_start = int(feature.location.start)
    exon_end = int(feature.location.end)
    strand = feature.location.strand if feature.location.strand is not None else 1

    return exon_seq, exon_start, exon_end, strand, feature

def scan_spcas9_sites(exon_seq: str, exon_start_0based: int, strand_label: str):
    """
    Returns candidates in the sequence orientation supplied.
    For a plus-strand scan, the guide sequence is reported as-is.
    For a minus-strand scan, the guide sequence is the reverse-complement derived spacer.
    """
    candidates = []
    seq = exon_seq.upper()
    L = len(seq)

    for i in range(L - 23 + 1):
        spacer = seq[i:i + 20]
        pam = seq[i + 20:i + 23]

        # SpCas9 PAM = NGG
        if not re.fullmatch(r"[ACGT]GG", pam):
            continue

        gc = round(gc_percent(spacer), 2)
        poly_t = has_poly_t(spacer)

        candidates.append({
            "Exon_Start_0based": exon_start_0based + i,
            "Exon_End_0based": exon_start_0based + i + 20,
            "Strand": strand_label,
            "gRNA_Sequence_5to3": spacer,
            "PAM": pam,
            "GC_%": gc,
            "Seed_GC_%": round(gc_percent(spacer[-10:]), 2),
            "Poly_T": "YES" if poly_t else "NO",
            "Homopolymer_Run": longest_homopolymer_run(spacer),
            "Heuristic_Rank_Score": heuristic_rank_score(spacer),
        })

    return candidates

def build_candidate_table():
    record = fetch_genbank_record(ACCESSION)
    exon_seq, exon_start, exon_end, exon_strand, exon_feature = extract_exon_sequence(record, EXON_NUMBER)

    # Scan the extracted exon sequence directly
    plus_candidates = scan_spcas9_sites(exon_seq, exon_start, "+")

    # Scan the reverse-complement of the exon sequence as well
    rc_seq = revcomp(exon_seq)
    minus_candidates = scan_spcas9_sites(rc_seq, exon_start, "-")

    # Basic filtering
    all_candidates = plus_candidates + minus_candidates
    df = pd.DataFrame(all_candidates)

    if df.empty:
        return df, record, exon_feature

    df = df[
        (df["GC_%"] >= MIN_GC) &
        (df["GC_%"] <= MAX_GC) &
        (df["Poly_T"] == "NO") &
        (df["Homopolymer_Run"] <= 4)
    ].copy()

    # Rank by score, then by closeness to 50% GC
    df["GC_Delta_50"] = (df["GC_%"] - 50.0).abs()
    df = df.sort_values(
        by=["Heuristic_Rank_Score", "GC_Delta_50"],
        ascending=[False, True]
    ).reset_index(drop=True)

    df.insert(0, "Rank", df.index + 1)
    return df, record, exon_feature

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    candidates_df, record, exon_feature = build_candidate_table()

    print(f"PCSK9 locus: {record.id}")
    print(f"Selected exon feature: {exon_feature.location}")
    print(f"Total candidates after filtering: {len(candidates_df)}")

    if not candidates_df.empty:
        print("\nTop 10 candidate guides:")
        print(candidates_df.head(10).to_string(index=False))

        out_csv = "PCSK9_exon1_spcas9_candidates.csv"
        candidates_df.to_csv(out_csv, index=False)
        print(f"\nSaved: {out_csv}")
    else:
        print("No candidates passed the filters.")

PCSK9 locus: NG_009061.1
Selected exon feature: [4929:5498](+)
Total candidates after filtering: 45

Top 10 candidate guides:
 Rank  Exon_Start_0based  Exon_End_0based Strand   gRNA_Sequence_5to3 PAM  GC_%  Seed_GC_% Poly_T  Homopolymer_Run  Heuristic_Rank_Score  GC_Delta_50
    1               5368             5388      - TCAGACCCTGAACTGAACGG CGG  55.0       50.0     NO                3                  96.0          5.0
    2               5425             5445      - TGCGGAAACCTTCTAGGGTG TGG  55.0       50.0     NO                3                  95.0          5.0
    3               4971             4991      + TCAAGCACCCACACCCTAGA AGG  55.0       50.0     NO                3                  94.0          5.0
    4               5236             5256      - AGGAGCTGAAGTTCAGGAGC AGG  55.0       60.0     NO                2                  94.0          5.0
    5               5426             5446      - GCGGAAACCTTCTAGGGTGT GGG  55.0       50.0     NO                3          